# 01 · Cleaning 10 years of Disneyland freeway traffic

**Goal:** turn Caltrans PeMS sensor files into one clean number per day: *cars exiting I-5 toward Disneyland*.

**Step 1.** Open one month of raw sensor data (July 2024) to see what it looks like. The files have no header row.

In [1]:
import pandas as pd
df = pd.read_csv("../data/pems/raw/d12_text_station_day_2024_07.txt.gz", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,07/01/2024 00:00:00,1201044,12,133,S,OR,NaN,2809,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07/01/2024 00:00:00,1201052,12,133,S,FR,NaN,2805,100,7008.0,NaN,NaN,NaN,NaN,NaN,NaN
2,07/01/2024 00:00:00,1201054,12,133,S,ML,1.285,8420,67,28581.0,0.0,0.0,0.0,0.3,4.7,10.9
3,07/01/2024 00:00:00,1201058,12,133,N,OR,NaN,2795,100,3423.0,NaN,NaN,NaN,NaN,NaN,NaN
4,07/01/2024 00:00:00,1201064,12,133,N,FR,NaN,2814,100,5687.0,NaN,NaN,NaN,NaN,NaN,NaN


**Step 2.** Name the columns, using the PeMS station-day file documentation.

In [2]:
df.columns = ["timestamp", "station", "district", 'route', 'direction', 'lane_type', 'station_length', 'samples' , 'pct_observed', 'total_flow', 'delay_35', 'delay_40', 'delay_45', 'delay_50', 'delay_55', 'delay_60']

Quick look after naming the columns.

In [5]:
df.head()

,timestamp,station,district,route,direction,lane_type,station_length,samples,pct_observed,total_flow,delay_35,delay_40,delay_45,delay_50,delay_55,delay_60
0,07/01/2024 00:00:00,1201044,12,133,S,OR,NaN,2809,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07/01/2024 00:00:00,1201052,12,133,S,FR,NaN,2805,100,7008.0,NaN,NaN,NaN,NaN,NaN,NaN
2,07/01/2024 00:00:00,1201054,12,133,S,ML,1.285,8420,67,28581.0,0.0,0.0,0.0,0.3,4.7,10.9
3,07/01/2024 00:00:00,1201058,12,133,N,OR,NaN,2795,100,3423.0,NaN,NaN,NaN,NaN,NaN,NaN
4,07/01/2024 00:00:00,1201064,12,133,N,FR,NaN,2814,100,5687.0,NaN,NaN,NaN,NaN,NaN,NaN


**Step 3.** Load the station metadata so each sensor ID has a name and type.

In [4]:
meta = pd.read_csv("../data/pems/raw/d12_text_meta_2023_12_05.txt", sep="\t")
meta.head()

,ID,Fwy,Dir,District,County,City,State_PM,Abs_PM,Latitude,Longitude,Length,Type,Lanes,Name,User_ID_1,User_ID_2,User_ID_3,User_ID_4
0,1201044,133,S,12,59,36770.0,9,8.991,33.661840,-117.755300,NaN,OR,1,BARRANCA2,565,NaN,NaN,NaN
1,1201052,133,S,12,59,36770.0,9,8.991,33.661840,-117.755300,NaN,FR,1,BARRANCA 2,565,NaN,NaN,NaN
2,1201054,133,S,12,59,36770.0,9,8.991,33.661840,-117.755300,1.285,ML,3,BARRANCA2,565,NaN,NaN,NaN
3,1201058,133,N,12,59,36770.0,8.866,8.857,33.659542,-117.756294,NaN,OR,1,BARRANCA1,551,NaN,NaN,NaN
4,1201064,133,N,12,59,36770.0,8.732,8.723,33.657392,-117.757636,NaN,FR,1,BARRANCA 1,551,NaN,NaN,NaN


**Step 4.** Find Disneyland's off-ramps: name contains *DISNEY* and type is `FR` (freeway off-ramp).

In [6]:
disney = meta[(meta["Name"].str.contains("DISNEY")) & (meta["Type"] == "FR")]
disney[["ID", "Dir", "Name"]]

,ID,Dir,Name
1619,1212201,S,HOV OFF DISNEYLAND
1620,1212202,S,DISNEY DR. OFFRAMP
1627,1212226,S,DISNEY WAY
1634,1212252,N,HOV DISNEY WAY OFF
1638,1212263,N,HOV DISNEY WAY OFF
1738,1212664,N,HOV DISNEY WAY OFF


**Step 5.** Keep only those off-ramp sensors and add a `date` column.

In [7]:
ids = disney["ID"].tolist()
d = df[df["station"].isin(ids)].copy()
d["date"] = pd.to_datetime(d["timestamp"]).dt.date
d[["date", "station", "total_flow", "pct_observed"]].head(12)


,date,station,total_flow,pct_observed
1619,2024-07-01,1212201,2368.0,100
1620,2024-07-01,1212202,10607.0,100
1627,2024-07-01,1212226,NaN,0
1634,2024-07-01,1212252,915.0,100
1638,2024-07-01,1212263,NaN,0
1738,2024-07-01,1212664,NaN,0
4206,2024-07-02,1212201,2468.0,100
4207,2024-07-02,1212202,10416.0,100
4214,2024-07-02,1212226,NaN,0
4221,2024-07-02,1212252,883.0,100


**Step 6.** First try: total cars per day. *Something looked wrong here: July 2024 traffic appeared to drop 93%.*

In [8]:
daily = d.groupby("date")["total_flow"].sum()
daily

date
2024-07-01    13890.0
2024-07-02    13767.0
2024-07-03    15914.0
2024-07-04    14191.0
2024-07-05    15267.0
2024-07-06    15201.0
2024-07-07    16354.0
2024-07-08    13561.0
2024-07-09    14145.0
2024-07-10    13364.0
2024-07-11    13389.0
2024-07-12    15845.0
2024-07-13    16729.0
2024-07-14    16978.0
2024-07-15    14229.0
2024-07-16    14369.0
2024-07-17    15370.0
2024-07-18    15023.0
2024-07-19     9239.0
2024-07-20      961.0
2024-07-21      971.0
2024-07-22     1000.0
2024-07-23      975.0
2024-07-24      948.0
2024-07-25      995.0
2024-07-26     1125.0
2024-07-27     1026.0
2024-07-28      874.0
2024-07-29      915.0
2024-07-30      931.0
2024-07-31     1024.0
Name: total_flow, dtype: float64

**Step 7.** Check sensor health. `pct_observed` is the % of the day a sensor actually reported. The "drop" was a sensor outage. **Missing data is not zero.**

In [9]:
health = d.pivot_table(index="date", columns="station", values="pct_observed")
health.iloc[15:25]

station,1212201,1212202,1212226,1212252,1212263,1212664
date,,,,,,
2024-07-16,100.0,100.0,0.0,100.0,0.0,0.0
2024-07-17,100.0,100.0,0.0,100.0,0.0,0.0
2024-07-18,100.0,100.0,0.0,100.0,0.0,0.0
2024-07-19,57.0,57.0,0.0,100.0,0.0,0.0
2024-07-20,0.0,0.0,0.0,100.0,0.0,0.0
2024-07-21,0.0,0.0,0.0,100.0,0.0,0.0
2024-07-22,0.0,0.0,0.0,100.0,0.0,0.0
2024-07-23,0.0,0.0,0.0,100.0,0.0,0.0
2024-07-24,0.0,0.0,0.0,100.0,0.0,0.0


**Step 8.** Rule: keep only the 2 main off-ramps, and only days where **both** reported 100% of the day.

In [10]:
main_ids = [1212201, 1212202]   # the 2 big Disneyland off-ramps
m = d[d["station"].isin(main_ids)]

good_days = m.groupby("date")["pct_observed"].min()
good_days = good_days[good_days == 100].index

daily = m[m["date"].isin(good_days)].groupby("date")["total_flow"].sum()
daily

date
2024-07-01    12975.0
2024-07-02    12884.0
2024-07-03    14856.0
2024-07-04    13489.0
2024-07-05    14428.0
2024-07-06    14481.0
2024-07-07    15495.0
2024-07-08    12577.0
2024-07-09    13239.0
2024-07-10    12423.0
2024-07-12    14735.0
2024-07-13    15750.0
2024-07-14    16046.0
2024-07-15    13302.0
2024-07-16    13471.0
2024-07-17    14289.0
2024-07-18    13911.0
Name: total_flow, dtype: float64

**Step 9.** Apply the same cleaning to all 129 monthly files (2016–2026).

In [11]:
import glob

files = sorted(glob.glob("../data/pems/raw/d12_text_station_day_*.txt.gz"))
print(len(files), "files found")

parts = []
for f in files:
    month = pd.read_csv(f, header=None, names=df.columns)
    parts.append(month[month["station"].isin(main_ids)])

allm = pd.concat(parts)
allm["date"] = pd.to_datetime(allm["timestamp"]).dt.date
allm.shape

129 files found


(7842, 17)

**Step 10.** Keep the healthy days (both sensors, 100% observed) and save `data/disney_offramp_daily.csv` (3,243 days), which is used by the SQL analysis and the Crowd Index.

In [12]:
health = allm.groupby("date")["pct_observed"].min()
count  = allm.groupby("date")["station"].nunique()
good_days = health[(health == 100) & (count == 2)].index

daily = allm[allm["date"].isin(good_days)].groupby("date")["total_flow"].sum().reset_index()
daily.to_csv("../data/disney_offramp_daily.csv", index=False)
print(len(daily), "clean days saved")
daily.tail()

3243 clean days saved


,date,total_flow
3238,2026-08-29,16248.0
3239,2026-08-30,15169.0
3240,2026-08-31,12449.0
3241,2026-09-01,13021.0
3242,2026-09-02,12941.0
